# OCR a ticker's filings — **one notebook, two machines**

Edit **cell 1** and run top to bottom. `ENVIRONMENT` is the only thing that decides where the
OCR happens; everything else means the same in both.

| `ENVIRONMENT` | the parse runs | the statement CSVs are written |
|---|---|---|
| `"LOCAL"` | on this machine's card (RTX 3050) | **as each quarter finishes** |
| `"KAGGLE"` | on a Kaggle T4, through `kgpu` | once, when the run folder is pulled back |

⚠️ **THE T4 IS NOT FASTER THAN THIS LAPTOP.** Measured twice, interleaved: local
**0.62-0.68 s/page** against T4 **0.78-0.95**. What Kaggle buys is a *second machine running in
parallel, free, without occupying this one* — that is worth having and it is not a multiplier.
CLAUDE.md §6-2-duodetricies.

## What an interrupted run leaves behind

With `MERGE_INTO_CSV = True` a **LOCAL** run upserts each quarter into
`raw_data/cafef/financials/statements/` the moment that quarter finishes, through
`pdf_ocr_merge` and its three refusals. `FinancialsBuilder._write` renders to a `.tmp` and
`os.replace`s it, and only the quarters a merge PRODUCED are rewritten — so stopping a 12-hour
run at hour 6 keeps every quarter that finished, unchanged, and can lose at most the one in
flight. A backup of all three CSVs is taken before the first write.

⚠️ **ON KAGGLE THAT GUARANTEE IS THE PULL's, NOT THE RUN's.** A kernel writes `/kaggle/working`
and exits; there is no path from it to this disk. The worker still writes one JSON per filing
as it goes, but the CSVs here are only touched after `kgpu pull` brings the folder home.

## ⚠️ How to tell whether it actually WROTE anything

**"The run finished" and "the CSV changed" are different facts, and only the second is the
point.** They came apart twice: HOSE_BSR and HOSE_CTG (**8 h 40 m, 201 accepted cells**) both
finished green, wrote complete run folders and created **no statement CSV at all** — every
statement refused for an empty `sane` band (`BND-1`). Nothing said so, because on KAGGLE the
merge runs on THIS machine after the pull while `metadata.json` is written by the worker,
which cannot reach this disk: `merged_into_csv` read `false` on every Kaggle run ever
(`MRG-1`).

The **last three cells** are the answer, in widening order of trust:

| cell | reads | answers |
|---|---|---|
| **the run folder** | `metadata.json` | what the parse found, and — since schema v3 — the `merge` block: how many statements were written, how many refused, and why |
| **refused / written** | the worker's `run.log` **and** the `merge` block | two different machines' decisions, under two headings, because conflating them once printed *"no refusals — every statement was accepted"* over a run that wrote nothing |
| **did it land?** | `raw_data/.../statements/*.csv` | the files themselves — `pdf` / `missing` / `cafef` per report, and how many rows this run put there |

## ⚠️ Three things this cannot do for you

1. **A statement a worker accepts is not always one a full run would.** `sane`'s magnitude band
   is reconstructed from the `pdf` rows on DISK here (`seed_history`) and accumulated IN THE RUN
   by `FinancialsBuilder.build()` — two populations, so the gate can disagree with itself.
   Measured on VIC Q3-2014.
2. ⚠️ **A TICKER WITH NOTHING ON DISK YET HAS NO BAND AT ALL, SO WITHOUT
   `FORCE_EMPTY_BAND` A GREEN RUN CREATES NO CSV.** Its first run is unguarded by
   construction and every merge of it is refused, so nothing is written, so the band is
   still empty next time — `BND-1` closes on itself. Measured on HOSE_BSR, 2026-08-30: a
   14-document Kaggle run finished clean, wrote a full run folder and not one statement
   CSV. `FORCE_EMPTY_BAND = True` breaks that loop for a ticker's FIRST run and lifts no
   other refusal; what it costs is the guard, so screen the artefact (the unit per report,
   total assets quarter on quarter) before quoting anything from a bootstrap run. The
   authoritative path for a new name is still a full Dagster `raw/cafef_financials` run,
   which accumulates its band as it goes.
3. ⚠️ **Nothing from a non-bank template may be quoted as a fundamental yet** — `CRP-1`.

In [16]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────────────
ENVIRONMENT = "LOCAL"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "CTG"          # ticker, as CafeF files it

# WHICH QUARTERS — YYYY-QQ. "2026-Q4" and the zero-padded "2026-04" are the same quarter.
#   ["2014-Q3"]              -> one quarter
#   ["2013-Q4", "2014-01"]   -> a batch, in any order
#   []  or  None             -> EVERY quarter this ticker files   (⚠️ 70+ documents, hours)
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file.
QUARTERS = ["2014-Q1"]        # ["2021-01"] is the same quarter, written the other way

# ⚠️ OVERWRITE decides two things, and they are one question: what do we do about a quarter
#    that is already on disk?
#   False -> FILL THE GAPS. A quarter already reading `pdf` in all three statements is dropped
#            before any OCR (and before it is uploaded), and a figure that DIFFERS from a good
#            `pdf` row is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
OVERWRITE = False

# UPSERT the accepted statements into raw_data/.../statements/*.csv.
#   LOCAL  -> one quarter at a time, as each finishes. This is the interruption guarantee.
#   KAGGLE -> once, after the pull. A kernel has no path to this disk.
# Either way it goes through `pdf_ocr_merge`, which BACKS THE THREE CSVs UP FIRST, prints every
# changed cell, and refuses FOUR things it cannot judge: a cumulative income statement
# whose missing quarters WERE filed (one whose priors never existed is written instead,
# carrying `months=6`/`12` — nothing can ever split it), a
# statement whose `sane` band was empty, a figure that DIFFERS from a good `pdf` row, and
# ⚠️ a document any of whose layers RAISED — an exception measures the MACHINE, not the
# filing, so whatever won the cascade won by default (`VCR-1`, 2026-08-29).
MERGE_INTO_CSV = True

# ⚠️ BOOTSTRAP A TICKER THAT HAS NO STATEMENT CSV YET — and it lifts a real guard.
#    `sane`'s magnitude band is rebuilt from the `pdf` rows ALREADY ON DISK
#    (`seed_history`), so a ticker parsed for the first time has NO band, `sane` fails
#    open, and the merge then refuses every statement the run produced. Nothing is
#    written, so the band stays empty and the next run refuses again — the loop closes
#    on itself. That is `BND-1`, and it is why a run can finish green and create no CSV.
#   True  -> write those statements anyway. This is the ONLY way a new ticker is ever
#            bootstrapped. The other three refusals are untouched: a cumulative income
#            statement, a figure that DIFFERS from a good `pdf` row, and a document
#            whose engine RAISED are still skipped and SAID.
#   False -> keep the guard. Correct for a ticker that already has history on disk,
#            where the band is real and a run that trips it is telling you something.
# ⚠️ What it costs: those figures passed NO magnitude check. Screen the artefact
#    before quoting any of them — the unit per report, and total assets quarter on
#    quarter. Both are read off the run folder alone: no PDF, no OCR, no network.
FORCE_EMPTY_BAND = True

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = False     # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the full 47-layer cascade, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)

## Resolving the inputs

Nothing below is edited. The cells validate the parameters, build the **one** input object the
CLI, this notebook and `kgpu` all share (`pdf_ocr_job.JobSpec`), and print what would run
before anything is spent.

In [17]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED, AND `import`
# WILL NOT SAY SO — once a module is in `sys.modules` the statement is a no-op, so re-running
# this notebook after the repo moves underneath it runs the OLD code. Measured 2026-08-30: a
# kernel started one commit earlier raised `AttributeError: module 'kgpu.runner' has no
# attribute 'RUN_STAGES'` in the RUN cell, while `utils.progress` — a module that same commit
# ADDED, so it had never been cached — imported fine two cells above.
# ⚠️ **THE LOUD FORM IS THE LUCKY ONE.** The silent form is an OCR run executing a previous
# commit's parser while `metadata.json` records HEAD's hash — a run folder that names code it
# did not run. So the repo's OWN packages are dropped here and re-imported from disk on every
# pass of this cell; third-party ones (torch, onnxruntime) are left alone, they do not move.
# ⚠️ It re-imports, so run this notebook TOP TO BOTTOM: objects built by an earlier pass
# belong to the module objects this line just discarded.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. `None` means every quarter.
QUARTERS = job.canonical_quarters(QUARTERS)

# The task label every progress line carries. ONE string, built once: LOCAL replaces it per
# document (`doc 2/3 HOSE_TCB Q3-2013`), KAGGLE keeps it for all six steps.
LABEL = f"{EXCHANGE}_{SYMBOL} " + (" ".join(QUARTERS) if QUARTERS else "all quarters")

print(f"environment : {ENVIRONMENT}")
print(f"ticker      : {EXCHANGE}_{SYMBOL}")
print(f"quarters    : {QUARTERS or 'ALL — every quarter this ticker files'}")
print(f"overwrite   : {OVERWRITE}"
      + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
print(f"upsert csv  : {MERGE_INTO_CSV}"
      + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
         else "   after the pull" if MERGE_INTO_CSV else ""))
print(f"bootstrap   : {FORCE_EMPTY_BAND}"
      + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
         "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
print(f"repo        : {REPO}")
print(f"cwd         : {Path.cwd()}")
print(f"code        : {REPO / 'src'}"
      + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
         if _RELOADED else "   (first import in this kernel)"))
# ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
# accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
# once, because it is on every line below it.
print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}"
      f"   ← overall %, a position in the plan")

environment : LOCAL
ticker      : HOSE_CTG
quarters    : ['2014-Q1']
overwrite   : False   (quarters already `pdf` in all three are skipped)
upsert csv  : True   per quarter, as each finishes
bootstrap   : True   an EMPTY `sane` band is written anyway — the only way a new ticker starts
repo        : d:\GIT\master-thesis
cwd         : d:\GIT\master-thesis\src\kaggle_gpu
code        : d:\GIT\master-thesis\src   (17 cached module(s) dropped, re-imported from disk)
log shape   :  33.7% - task - sub-task - detail   ← overall %, a position in the plan


In [18]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL
# run and a KAGGLE run of the same parameters are the same procedure on two machines, not two
# procedures. What differs is the stack, and every run records its `stack_fingerprint`.
SPEC = CFG = PREPARED = None

if ENVIRONMENT == "LOCAL":
    SPEC = job.JobSpec(
        exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
        force_empty_band=FORCE_EMPTY_BAND,
        notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
    )
    # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
    # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
    # quarter you asked for is already parsed and OVERWRITE is False.
    PREPARED = SPEC.prepare()
    print("\n".join(PREPARED.describe()))
    print()
    for _t in PREPARED.tasks:
        print(f"  {_t.period:<8} {_t.file[:56]:<56} "
              f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
              + ("  CUMULATIVE" if _t.cumulative else ""))
    if PREPARED.template != "bank":
        print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
              f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
              f"`assets == resources` — true by\n   construction on any page that reads both. "
              f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
else:
    from kgpu import pdf_ocr, runner                 # noqa: E402

    CFG = pdf_ocr.job(
        SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
        # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes
        # /kaggle/working and exits — so this is the PULL's knob, read by
        # `runner.merge_statements` on this machine after the folder lands.
        force_empty_band=FORCE_EMPTY_BAND,
    )
    print("\n".join(pdf_ocr.describe(CFG)))
    print()
    # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
    # payload cannot diverge from the worker's own choice.
    runner.plan(CFG)

symbol       : HOSE_CTG
template     : bank   (detect_template (CafeF fingerprint, over the network))
documents    : 1  (Q1-2014)
quarters     : ['2014-Q1']   selected ['2014-Q1']
skipped      : 0 quarter(s) already `pdf` in all three statements
cascade      : 49 of 49 layers
data root    : D:\GIT\master-thesis\raw_data\cafef
models       : det=deepdoc_det.onnx vietocr=vgg_seq2seq.pth
vietocr cfg  : D:\GIT\master-thesis\src\web_scraper\models\vietocr_vgg_seq2seq.yml

  Q1-2014  Q1-2014_bao_cao_tai_chinh_hop_nhat_quy_1_nam_2014.pdf       4.3 MB


In [19]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ **THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE.** A rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists — and until 2026-08-29
# this cell called `rehearse` straight after `plan`, which raises `no staged payload` on every
# first run of a job and points at `python -m kgpu data <job>`, a command that cannot resolve
# a job this notebook COMPUTED rather than wrote into kaggle_config.json.
# `export` is local and free — it writes the zip, it does not upload; the RUN cell below
# re-exports and uploads (`refresh_data=True`), so nothing here commits you to anything.
#
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. **An empty band is the warning to stop for**: `sane` fails open without one, and
# that is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export                       # noqa: E402

    # Two steps, one line each, in the same shape the RUN cell prints — `capture()` re-emits
    # `export`'s and `rehearse`'s own output as the DETAIL of the step that produced it.
    DRESS = progress.Stages([("export", "stage payload", 1.0),
                             ("rehearse", "rehearse worker", 1.0)], label=LABEL)
    DRESS.begin("export", "local, no upload, no quota")
    with DRESS.capture():
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    DRESS.begin("rehearse", "both Kaggle mount layouts")
    with DRESS.capture():
        runner.rehearse(CFG)
    DRESS.done("rehearsed — nothing was spent")
else:
    print("skipped" if ENVIRONMENT == "KAGGLE" else "LOCAL — nothing to rehearse")

LOCAL — nothing to rehearse


In [20]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ **EVERY LINE BELOW HAS THE SAME SHAPE ON BOTH MACHINES:**
#
#     ` 33.7% - <task> - <sub-task> - <detail>`
#
# LOCAL, the task is the DOCUMENT and the sub-task its position in the cascade
# (`doc 2/3 HOSE_TCB Q3-2013 - layer 12/49 onnx@300 - page 40/96`); KAGGLE, the task is the
# STEP of the round trip and the sub-task the step's name (`step 4/6 … - wait kernel - …`).
# One formatter, `utils.progress`, so the two cannot drift.
#
# ⚠️ **THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME.** A filing
# accepted at its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33 min,
# so the number is a LOWER BOUND on real progress — a run finishes early, it does not stall
# at 99 %. (LOCAL the within-document denominator is OCR PASSES, not the 49 cascade
# positions: half the layers only re-map a parse the cache already holds.)
# ⚠️ And on KAGGLE it stands still through `wait kernel` unless this exact job has completed
# once before: `kernels_status` reports QUEUED / RUNNING / COMPLETE and no fraction, so the
# only honest clock is this job's own last duration. The detail keeps printing the minutes.
#
# ⚠️ Each document's JSON is written BEFORE the next one starts, and — LOCAL, with
# MERGE_INTO_CSV — each quarter is upserted into the statement CSVs before the next document is
# opened. Single filings here cost over half an hour; a run that kept its results in memory
# would lose every one of them to the first interrupt.
#
# ⚠️ Budget: a filing accepted at layer 1 is ~1 min, one that defeats the whole cascade was
# 26-33 min when measured, and Kaggle adds a ~5 min QUEUE before anything starts.
FOLDER = EXIT = REPORT = None

if not EXECUTE:
    print("EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL":
    # `job.run` prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    FOLDER = job.run(SPEC)
else:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup taken first\n"
              "    and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    REPORT = progress.Stages(runner.RUN_STAGES, label=LABEL)
    EXIT = runner.run(CFG, refresh_data=True, progress=REPORT)
    REPORT.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")

web_scraper.pdf_ocr_job
started   2026-08-30 13:39:48 GMT+7
device    cuda
gpu       NVIDIA GeForce RTX 3050 Laptop GPU  |  4,096 MiB total, 3,303 MiB free  |  CUDA 12.1, torch 2.5.1+cu121   [torch]
  0.0% - HOSE_CTG - plan - overall %    : documents finished + OCR passes done of the 7 this 49-layer cascade can cost, over 1 document(s) — a POSITION IN THE PLAN, never a fraction of the time
  0.0% - HOSE_CTG - plan - symbol       : HOSE_CTG
  0.0% - HOSE_CTG - plan - template     : bank   (detect_template (CafeF fingerprint, over the network))
  0.0% - HOSE_CTG - plan - documents    : 1  (Q1-2014)
  0.0% - HOSE_CTG - plan - quarters     : ['2014-Q1']   selected ['2014-Q1']
  0.0% - HOSE_CTG - plan - skipped      : 0 quarter(s) already `pdf` in all three statements
  0.0% - HOSE_CTG - plan - cascade      : 49 of 49 layers
  0.0% - HOSE_CTG - plan - data root    : D:\GIT\master-thesis\raw_data\cafef
  0.0% - HOSE_CTG - plan - models       : det=deepdoc_det.onnx vietocr=vgg_seq2seq.pth
  0

d:\GIT\master-thesis\mt_env\Lib\site-packages\gdown\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
d:\GIT\master-thesis\mt_env\Lib\site-packages\vietocr\tool\predictor.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explic

  1.8% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 7/55  12% of this pass   ~77 s left
  3.4% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 13/55  23% of this pass   ~56 s left
  4.9% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 19/55  34% of this pass   ~44 s left
  6.5% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 25/55  45% of this pass   ~35 s left
  8.1% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 31/55  56% of this pass   ~27 s left
  9.6% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 37/55  67% of this pass   ~20 s left
 11.2% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 43/55  78% of this pass   ~13 s left
 12.7% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 49/55  89% of this pass   ~7 s left
 14.3% - doc 1/1 HOSE_CTG Q1-2014 - layer 1/49 onnx@200 - page 55/55 100% of this pass   ~0 s left
 14.3% - doc 1/1 HOSE_CTG Q1-2014 - layer 2/49 onnx@300 - OCR pass 2/7
 14.5% - doc 1/1 HOSE_CTG Q1-201

## The result

`compare()` scores every parsed cell against the statement CSV on disk, so a verdict means:

| verdict | what it says |
|---|---|
| `REPRODUCED` | every cell, the winning **layer**, the unit and `publish_date` match disk |
| `DIFFERS` | one of those moved — the run names which, with both figures |
| `absent in this run` | the cascade refused the statement; the log below says why |
| `no pdf row on disk to compare against` | ⚠️ **a RECOVERY, not a reproduction** — nothing scored it |
| *(refused)* | a cumulative income statement is not scored against a row covering a different span; two rows whose `months` agree are compared normally |

⚠️ **READ THE FIRST REFUSAL, NOT THE LAST.** A cascade's final refusal names the hardest path
tried, not the blocking defect — the label `fx not mapped` sent this repo down a wrong
diagnosis for two days, and six of the seven quarters blamed on it then parsed at a **strict**
layer with no FX change at all (CLAUDE.md §6-2-duovicies).

In [21]:
# ── READ THE RUN FOLDER ─────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
import json                                          # noqa: E402

PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)
LATEST = FOLDERS[-1] if FOLDERS else None
META = MERGE = None

if LATEST is None:
    print(f"no run folder matching {PATTERN}")
else:
    META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
    SCHEMA = META.get("schema_version", 1)
    print(LATEST.name)
    print(f"  commit       : {META.get('git_commit')}")
    # ⚠️ §5 rule 2 at the artefact: an older run folder carries none of the fields below, and
    # printing `None` for them would read as a VALUE rather than as "this run predates the
    # field". `schema_version` is what tells the two apart.
    V2 = SCHEMA >= 2
    OLDER = "— (schema v1: this run predates the field)"
    print(f"  filter       : quarters={inputs.get('quarters')}  "
          f"periods={inputs.get('periods')}  "
          f"overwrite={inputs.get('overwrite') if V2 else OLDER}")
    print(f"  skipped      : {inputs.get('skipped_already_parsed') if V2 else OLDER}")
    print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
    # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
    # because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
             if ocr.get("pin_violations") else ""))

    # ⚠️ **THE UPSERT IS THE ONE FIELD THE PARSING PROCESS CANNOT KNOW.** `metadata.json` is
    # written by whatever ran the OCR, and on KAGGLE that is a worker with no path to this
    # disk — so `merged_into_csv` read `false` on every Kaggle run ever, whatever the pull did
    # with it. Since schema v3 the merge writes its own outcome back
    # (`pdf_ocr_merge.record_merge`), and an ABSENT block on an older folder means "this run
    # predates the field", never "nothing was written".
    MERGE = META.get("merge")
    if MERGE:
        print(f"  upserted     : {MERGE['statements_written']} statement(s) written, "
              f"{MERGE['statements_skipped']} refused"
              + (f"   backup={inputs.get('merge_backup')}"
                 if inputs.get("merge_backup") else ""))
        if MERGE["periods_written"]:
            got = MERGE["periods_written"]
            print(f"                 {len(got)} quarter(s): "
                  f"{', '.join(got[:8])}{' …' if len(got) > 8 else ''}")
    elif SCHEMA >= 3:
        print("  upserted     : ⚠️ NOTHING — no merge ran against this run folder.")
    else:
        print(f"  upserted     : — (schema v{SCHEMA} predates the `merge` block; "
              f"`inputs.merged_into_csv` says {inputs.get('merged_into_csv')}, "
              f"which on a KAGGLE run was always false)")

    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in META.get("results", []):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    PER_DOC = {r["period"]: r["seconds"] for r in META.get("results", [])}
    print(chr(10) + f"  parse: {sum(PER_DOC.values()) / 60:.1f} min over "
          f"{len(PER_DOC)} document(s)")


20260830-133945__hose_ctg__pdf_ocr
  commit       : 45d34404
  filter       : quarters=['2014-Q1']  periods=None  overwrite=False
  skipped      : []
  template     : bank  (detect_template (CafeF fingerprint, over the network))
  detection    : CUDAExecutionProvider   (onnxruntime 1.22.0)
  recognition  : cuda
  stack        : e6b778e294b4
  upserted     : 2 statement(s) written, 1 refused   backup=D:\GIT\master-thesis\raw_data\_backup\statements\20260830-135143__HOSE_CTG
                 1 quarter(s): Q1-2014

  period     report             layer                          items  status   verdict
  Q1-2014    balance_sheet      onnx@200                          53  pdf      no pdf row on disk to compare against
  Q1-2014    income_statement   onnx@300                          20  pdf      REPRODUCED
  Q1-2014    cash_flow          —                                  0  absent   absent in this run

  parse: 11.9 min over 1 document(s)


In [22]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
# Two different questions, from two different places, and conflating them is how an 8-hour
# run came to look like a success while writing no CSV at all:
#
#   the PARSE refused a statement   -> `run.log`, written by whatever ran the OCR
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
#
# ⚠️ **ON KAGGLE THOSE ARE TWO MACHINES.** The worker's `run.log` cannot hold a merge
# decision — the merge happens here, after the pull — so a cell that greps that log for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print
# "no refusals — every statement was accepted". That is a false success claim, and it was
# printed over a run that wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
#
# ⚠️ **MATCH ON THE DETAIL, NOT ON THE START OF THE LINE.** Since 2026-08-30 every line in
# `run.log` is ` xx.x% - task - sub-task - detail`, so a filter anchored at the start of the
# line matches nothing — `progress.detail_of` is the segment that used to BE the line.
if LATEST is not None:
    LOG = (LATEST / "run.log").read_text(encoding="utf-8", errors="replace")
    HITS = [ln for ln in LOG.splitlines()
            if "absent after" in ln or "reconcile:" in ln or "sane:" in ln]
    print("── the PARSE refused ────────────────────────────────────────")
    print(chr(10).join(HITS) if HITS else
          "  nothing — every statement the cascade opened was accepted")

    print(chr(10) + "── the MERGE decided ───────────────────────────────────────")
    if MERGE:
        for ev in MERGE["events"]:
            for d in ev["decisions"]:
                mark = "WRITE " if d["action"] == "write" else "skip  "
                items = f"[{d['layer']}] {d['items']} items" if d["layer"] else ""
                print(f"  {mark} {d['period']:9} {d['report']:18} {items:32} {d['reason']}")
        print(chr(10) + f"  -> {MERGE['statements_written']} written, "
              f"{MERGE['statements_skipped']} refused")
        if not MERGE["statements_written"]:
            print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the "
                  "commonest reason is an")
            print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True and merge again. "
                  "It lifts ONE guard")
            print("     and no other, so screen the artefact before quoting anything "
                  "(`BND-1`).")
    else:
        # A LOCAL run before schema v3 wrote its merge decisions into `run.log` instead.
        OLD = [ln for ln in LOG.splitlines()
               if progress.detail_of(ln).startswith(
                   ("WRITE ", "skip ", "backup:", "written:"))]
        if OLD:
            print(chr(10).join(OLD))
        else:
            print("  ⚠️ NO MERGE RAN against this run folder — the statement CSVs "
                  "were not opened.")
            print("     KAGGLE: `python -m kgpu merge <job>` finishes it "
                  "(--force-empty-band for a")
            print("     ticker with no CSV yet).   LOCAL: set MERGE_INTO_CSV = True.")


── the PARSE refused ────────────────────────────────────────
100.0% - doc 1/1 HOSE_CTG Q1-2014 - layer 49/49 onnx@300+pad6+annual+extra - WARNING:     cash_flow absent after 49 layer(s):
100.0% - doc 1/1 HOSE_CTG Q1-2014 - layer 49/49 onnx@300+pad6+annual+extra - WARNING:       [onnx@200] reconcile: no closing cash balance

── the MERGE decided ───────────────────────────────────────
  WRITE  Q1-2014   balance_sheet      [onnx@200] 53 items              recovers a quarter disk records as `missing`
  skip   Q1-2014   cash_flow                                           absent in this run
  WRITE  Q1-2014   income_statement   [onnx@300] 20 items              same figures, same layer — recording the span this row covers (`months`: unrecorded -> 3)

  -> 2 written, 1 refused


In [23]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ **THE ONE QUESTION THIS NOTEBOOK EXISTS TO ANSWER, ASKED OF THE FILES AND NOT OF A
# LOG.** Everything above reports what some process DECIDED; this reads what is on disk. The
# two came apart on a Kaggle round trip that finished green, wrote a complete run folder and
# created no CSV at all (`BND-1`, measured on HOSE_BSR and again on HOSE_CTG) — and no amount
# of log reading would have said so, because the merge that refused everything ran on the
# other machine.
import csv                                            # noqa: E402

from web_scraper import cafef_financials as fin       # noqa: E402

# ⚠️ **`CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT.** `statement_path()` reads
# `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while cell 3
# `os.chdir`s to `src/kaggle_gpu`, because that is where `kgpu` stages its payload. So the
# first version of this cell reported `NO FILE` for a ticker whose three CSVs were on disk.
# Re-pointing at the repo is what `pdf_ocr_merge.plan_merge` does for the same reason; the
# path is PRINTED because a directory nobody names is a directory nobody checks.
ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
# ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
# skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance
# sheet was `identical to the row already on disk` — so a period-only set credits this run
# with a row it deliberately left alone.
MINE = {(d["period"], d["report"])
        for ev in (MERGE or {}).get("events", [])
        for d in ev["decisions"] if d["action"] == "write" and ev["applied"]}
if TPL is None:
    print("no template resolved — run the cells above first")
else:
    print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
    print()
    ANY = False
    for _report in fin.REPORTS:
        _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
        if not _path.is_file():
            print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
            continue
        ANY = True
        with open(_path, encoding="utf-8-sig") as _f:
            _rows = list(csv.DictReader(_f))
        _src = {}
        for _r in _rows:
            _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
        _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                 and (_r["period"], _report) in MINE]
        print(f"  {_report:18} {len(_rows):>3} quarters   "
              + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
              + (f"   <- {len(_mine)} from this run" if _mine else ""))
        # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
        # A `cafef` row is an HTML transcription and must not be in this file.
        if _src.get("cafef"):
            print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                  f"transcription. §5 rule 24 forbids it.")
    if not ANY:
        print()
        print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
              "and")
        print("     nothing was upserted — the MERGE section above says which refusal "
              "stopped it.")


D:\GIT\master-thesis\raw_data\cafef\financials\statements\bank   HOSE_CTG

  balance_sheet       70 quarters   missing=39  pdf=31   <- 1 from this run
  income_statement    70 quarters   missing=35  pdf=35   <- 1 from this run
  cash_flow           70 quarters   missing=9  pdf=61
